# 04. 활동 위축 탐지 규칙 (총 활동량 I1)
> 계획서: `계획/P4_활동위축_탐지규칙.md` (Codex 1·2·3차 검증 반영) · 수식은 계획서 4b 와 `code/detect_cnt.py` 에 그대로 구현되어 있다.

**이 노트북이 하는 일** — 국내 5채널 총 활동량 I1의 변화율로 '활동 위축'을 탐지하는 규칙을 정한다. **입금은 쓰지 않는다** (P5의 검증이 순환이 되지 않도록).

**실행 순서(고정)**: ① 계절성 → ② 비교 기준(전년 같은 달 / 직전 기간) → ③ 창 길이 k → ④ 활동량 하한 Φ → ⑤ 판정 가능 집합 → ⑥ 탐지 임계 n → ⑦ 최종 탐지.
- 규칙은 **P36에서 한 번 정해 고정**하고, **P35만**(독립 재현)과 **P35+**(포함 범위 민감도)에는 같은 규칙을 적용해 결정을 비교한다.
- 이 규칙은 36개월 자료 전체로 정한 **사후 고정 규칙**이다. 실시간 운영 성능이 아니라 '이 기간의 후행 연관'을 본다.
- 결론은 **활동량 하한 이상 기업**에 한정한다.

**화면 출력은 집계값뿐** · 5 미만 칸은 `<5` · 법인ID 없음.

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 설정: 여기만 고치세요 ────────────────────────────────────────────
WORK_DIR = os.environ.get("CNT_WORK_DIR") or None   # 기본값: 레포의 data/work_cnt (.gitignore 대상)
# ────────────────────────────────────────────────────────────────────

def _find_lib():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "code", base / "analysis" / "code"):
            if (cand / "cntlib.py").exists():
                return cand
    raise SystemExit("cntlib.py 를 찾지 못했습니다. 노트북을 analysis/notebooks 에서 여세요.")

sys.path.insert(0, str(_find_lib()))
import cntlib as cl
cl.setup_korean_font()
WORK = cl.resolve_work_dir(WORK_DIR)
OUT = WORK / "output"
print("산출물 폴더:", WORK)

In [ ]:
panel = cl.CountPanel.load(WORK / "interim" / "count_panel.npz")
_m = np.load(WORK / "interim" / "sample_masks.npz")
masks = {k: _m[k] for k in _m.files}
if panel.synthetic:
    print(cl.SYNTHETIC_NOTICE)
pops = {"P36": masks["P36"], "P35+": masks["P35p"]}
print({k: int(v.sum()) for k, v in pops.items()}, "개 기업")

import detect_cnt as dc

POP = {"P36": masks["P36"], "P35만": masks["P35"], "P35+": masks["P35p"]}
I1 = {name: dc.i1_matrix(panel, m) for name, m in POP.items()}
print({k: int(v.sum()) for k, v in POP.items()}, "개 기업")
print("규칙은 P36에서 정하고 P35만·P35+에는 같은 규칙을 적용합니다. 입금은 이 노트북에서 쓰지 않습니다.")

## 1. 첫 표: k별·비교 방식별 판정 가능 월과 사건 수, 검정력 (계획서 4d)
36개월 자료에서는 탐지(k+12 이후)와 사후 6개월 추적이 모두 필요해 탐지 가능 월이 짧습니다. 각 칸은 그 조합 **자신의** Φ와 n으로 계산한 값입니다.
- N1 = 탐지 기업 수, N0 = 판정 가능 기업 중 탐지되지 않은 기업 수, **MDE** = 대조군 약화율 5%·설계 효과 1.5 가정에서 80% 검정력으로 검출 가능한 최소 비율비.

In [ ]:
def _fmt(d):
    d = d.copy()
    for c in ["판정 가능 기업-월", "N1 (임시 30%)", "N1 (최종 n)", "판정 가능 기업", "N0"]:
        if c in d:
            d[c] = d[c].map(lambda x: cl.sup(x) if pd.notna(x) else "-")
    for c in ["판정 가능 %", "(a) 비교 평균 0 %", "(b) 하한 미만 %", "(c) 빈 달 부족 %"]:
        d[c] = d[c].round(1)
    d["Φ"] = d["Φ"].round(2)
    d["MDE"] = d["MDE"].map(lambda x: "-" if pd.isna(x) else ("검출 불가" if x == float("inf") else f"{x:.2f}"))
    return d

t4d = pd.concat([dc.first_table(I1["P36"], "P36"), dc.first_table(I1["P35만"], "P35만")], ignore_index=True)
cl.show_table(_fmt(t4d), "k별·비교 방식별 판정 가능 월·사건 수·검정력", OUT / "nb04", "first_table_4d")

## 2. ①~⑥ 결정 (세 표본에서 같은 절차를 돌려 비교; 고정 규칙은 P36)

In [ ]:
DEC = {name: dc.decide(I1[name]) for name in POP}
RULE = DEC["P36"]
MODE, K, L, PHI, N = RULE["mode"], RULE["k"], RULE["L"], RULE["phi"], RULE["n"]

def _ratio(x):
    return "-" if x is None or x == float("inf") else round(x, 2)

rows = []
for name, d in DEC.items():
    rows.append({"표본": name, "① 연도 쌍 상관 평균": round(d["sea"]["corr_mean"], 3), "연도별 최대/최소 평균": round(d["sea"]["maxmin_mean"], 3),
                 "계절성": "있음" if d["sea"]["flag"] else "없음",
                 "② 월별 탐지율 최대/최소 (전년)": _ratio(d["mode_info"]["ratio"]["전년"]), "(직전)": _ratio(d["mode_info"]["ratio"]["직전"]),
                 "비교 기준": d["mode"], "③ k": d["k"], "④ Φ": round(d["phi"], 2), "⑥ n (%)": d["n"]})
cl.show_table(pd.DataFrame(rows), "①~⑥ 결정: 표본별 (P36 이 고정 규칙)", OUT / "nb04", "decisions")
print(f"\n고정 규칙 (P36): 비교 기준 = {MODE}, k = {K}개월, 비교 간격 L = {L}, 활동량 하한 Φ = {PHI:.2f}, 탐지 임계 n = {N:.0f}%")
same = all((DEC[n_]["mode"], DEC[n_]["k"], DEC[n_]["n"]) == (MODE, K, N) for n_ in ("P35만", "P35+"))
print("P35만·P35+ 의 (비교 기준, k, n) 이 P36과 모두 같은가:", "예" if same else "아니오 (위 표에서 차이 확인)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for y, v in zip((2023, 2024, 2025), RULE["sea"]["index"]):
    ax[0].plot(range(1, 13), v, marker="o", label=str(y))
ax[0].set_title("월 지수 (그 달 I1 중앙값 ÷ 그 해 월 중앙값 평균), P36"); ax[0].set_xlabel("월"); ax[0].legend()
x = list(range(3 + 12, 31))
for m_ in dc.MODES:
    ax[1].plot(x, [r_ * 100 for r_ in RULE["mode_info"]["rates"][m_]], marker=".", label=m_)
ax[1].set_title("후보월별 탐지율 (임시 30%, k=3), P36"); ax[1].set_xlabel("후보월 t"); ax[1].set_ylabel("%"); ax[1].legend()
plt.tight_layout(); plt.show()

### ③ 창 길이: 일시적 하락 비율
탐지 기업 중 **사후 k개월 변화율(작년 같은 달 대비)이 −10% 이상으로 회복**한 비율입니다. 20% 미만인 가장 짧은 창을 택합니다 (계획서 4b).

In [ ]:
rows = []
for name, d in DEC.items():
    for s_ in d["shares"]:
        rows.append({"표본": name, "비교 기준": d["mode"], "k": s_["k"], "탐지 기업 (임시 30%)": cl.sup(s_["detected"]), "사후 정의 기업": cl.sup(s_["defined"]),
                     "일시적 하락 %": round(s_["share"] * 100, 1) if s_["defined"] else None, "선택": "◀" if s_["k"] == d["k"] else ""})
cl.show_table(pd.DataFrame(rows), "일시적 하락 비율 (k별)", OUT / "nb04", "transient_share")
for name, d in DEC.items():
    near = [s_["k"] for s_ in d["shares"] if s_["defined"] and abs(s_["share"] - 0.20) <= 0.03]
    print(f"{name}: k = {d['k']} ({d['k_why']})" + (f"  ※ 20% 경계 근처(±3%p)인 k: {near}" if near else ""))

## 3. ④ 활동량 하한 Φ와 제외 비율 (P36)
비교 평균 C를 값 기준 10분위로 나누고(경계가 같으면 합치고, 분위당 100 기업-월 미만이면 병합) 분위별 변화율 IQR을 봅니다. **모든 상위 분위가 기준(상위 절반 IQR 중앙값 × 1.5) 이하**가 되는 가장 낮은 분위의 하한이 Φ입니다.

In [ ]:
info = RULE["phi_info"]
tab = pd.DataFrame({"분위(병합 후)": range(1, info["bins"] + 1), "C 하한": info["lo"].round(2), "C 상한": info["hi"].round(2),
                    "기업-월": [cl.sup(x) for x in info["count"]], "변화율 IQR": info["iqr"].round(3),
                    "기준 이하": ["예" if p else "아니오" for p in info["passes"]], "Φ 시작": ["◀" if i == info["d_star"] else "" for i in range(info["bins"])]})
cl.show_table(tab, f"분위별 변화율 IQR (기준 = 1.5 × {info['ref']:.3f} = {1.5 * info['ref']:.3f})", OUT / "nb04", "phi_deciles")

ch = RULE["change"]; lo = K + 11
Cw, rw = ch.C[:, lo:30], ch.r[:, lo:30]
firm_ix = np.repeat(np.arange(Cw.shape[0]), Cw.shape[1]).reshape(Cw.shape)
bs = dc.phi_bootstrap(firm_ix.ravel(), Cw.ravel(), rw.ravel(), B=200)
bs = bs[np.isfinite(bs)]
print(f"\nΦ = {PHI:.2f}.  기업 단위 재표집 200회의 Φ 분포: 중앙값 {np.median(bs):.2f}, 5~95분위 {np.percentile(bs, 5):.2f} ~ {np.percentile(bs, 95):.2f}, 범위 {bs.min():.2f} ~ {bs.max():.2f}  (보고용, Φ는 바꾸지 않음)")
st = dc.status_matrix(ch, PHI)[:, lo:30]
defined_cd = (~np.isnan(rw)).sum()
print(f"하한으로 제외된 비율(변화율이 정의된 기업-월 중 C < Φ): {(st == 2).sum() / max(defined_cd, 1) * 100:.1f}%   (판정 가능 {(st == 0).sum():,} 기업-월)")

## 4. ⑥ 탐지 임계 n (P36)
판정 가능 기업-월 변화율 r의 **5번째 백분위수**로 n₀ = −100×Q₀.₀₅ (양의 %)를 구하고 5%p 단위로 반올림(0.5 올림, 최소 5)합니다.

In [ ]:
jr = np.where(~np.isnan(rw) & (Cw >= PHI), rw, np.nan)
d_ = dc.describe_r(jr)
tab = pd.DataFrame([{"관측": cl.sup(d_["관측"]), "평균": round(d_["평균"], 3), "표준편차": round(d_["표준편차"], 3), **{k_: round(d_[k_], 3) for k_ in ["1%", "5%", "10%", "25%", "50%", "75%", "90%"]}, "왜도": round(d_["왜도"], 2)}])
cl.show_table(tab, "판정 가능 기업-월 변화율 r 의 분포 (P36)", OUT / "nb04", "r_distribution")
ni = RULE["n_info"]
print(f"5번째 백분위수 = {ni['q05']:.3f} → n₀ = {ni['n0']:.1f}% → n = {ni['n']:.0f}%.   r 이 정확히 −n% 인 기업-월 비율: {ni['exact_share'] * 100:.2f}%" + ("  ※ 1% 초과: n±5%p 경계 민감도를 함께 봅니다." if ni["exact_share"] > 0.01 else ""))
rows = []
for nn_ in (N - 5, N, N + 5):
    if nn_ < 5:
        continue
    dd = dc.first_detect(ch, PHI, nn_ / 100.0)
    rows.append({"n (%)": nn_, "탐지 기업 N1": cl.sup((dd >= 0).sum()), "구분": "고정 규칙" if nn_ == N else "경계 민감도"})
cl.show_table(pd.DataFrame(rows), "n ±5%p 에서의 탐지 기업 수 (P36)", OUT / "nb04", "n_sensitivity")

## 5. ⑦ 최종 탐지: 고정 규칙을 세 표본에 적용

In [ ]:
APPLY = {name: dc.apply_rule(I1[name], MODE, K, PHI, N) for name in POP}
rows = []
for name, a in APPLY.items():
    jf = int((a["status"] == 0).any(axis=1).sum()); n1 = int((a["det"] >= 0).sum())
    rows.append({"표본": name, "기업 수": cl.sup(len(a["det"])), "판정 가능 기업": cl.sup(jf), "탐지 기업 N1": cl.sup(n1), "탐지율 % (판정 가능 기업 중)": round(100 * n1 / max(jf, 1), 1)})
cl.show_table(pd.DataFrame(rows), "고정 규칙 적용 결과 (M0: 빈 달은 채우지 않음)", OUT / "nb04", "detect_counts")

det36 = APPLY["P36"]["det"]
rows = [{"기준월": int(panel.months[e]), "처음 탐지된 기업": cl.sup((det36 == e).sum())} for e in dc.e_range(K)]
cl.show_table(pd.DataFrame(rows), "월별 최초 탐지 기업 수 (P36)", OUT / "nb04", "first_detect_by_month")

### M1(인과적 채움) 민감도 — P35만
후보월마다 **그 달까지의 관측값만**으로 과거 빈 달을 채워 탐지 이력을 다시 계산합니다. 문턱(Φ, n)은 P36에서 정한 값 그대로입니다. 채움은 정확도(P1: ±1 코드 이내 약 94%)가 탐지 정확도는 아니므로 **최초 탐지 일치율**을 함께 봅니다.

In [ ]:
t0 = time.time()
det0 = APPLY["P35만"]["det"]
det1 = dc.first_detect_m1(panel, POP["P35만"], K, L, PHI, N / 100.0)
d0, d1 = det0 >= 0, det1 >= 0
either = d0 | d1
same_month = (det0 == det1) & d0 & d1
changed = (d0 != d1) | (d0 & d1 & (det0 != det1))
tab = pd.DataFrame([{"표본": "P35만", "기업 수": cl.sup(len(det0)), "M0 탐지 기업": cl.sup(d0.sum()), "M1 탐지 기업": cl.sup(d1.sum()),
                     "둘 중 하나라도 탐지된 기업": cl.sup(either.sum()), "최초 탐지월까지 일치 %": round(100 * same_month.sum() / max(either.sum(), 1), 1),
                     "판정 변경률 % (전체 기업 중)": round(100 * changed.mean(), 1)}])
cl.show_table(tab, "M0 vs M1 (P35만)", OUT / "nb04", "m1_vs_m0")
print(f"({time.time() - t0:.0f}초)")

## 6. 판정 불가율과 적용 범위 (P36 · P35+)
판정 불가 3종을 정상으로 세지 않고 **월·업종군·규모(2023년 평균 I1 5분위, 탐지 창보다 앞선 기간)별로** 봅니다. (a) 비교 평균 0, (b) 활동량 하한 미만, (c) 빈 달 부족.

In [ ]:
import warnings
lo = K + 11
levels = panel.attr_levels["업종_대분류"]
sec = {name: dc.firm_attribute(panel, m, "업종_대분류") for name, m in POP.items()}
cnt36 = np.bincount(sec["P36"][sec["P36"] >= 0], minlength=len(levels))
TOP5 = [int(x) for x in np.argsort(-cnt36)[:5]]
SG = [(g, levels[g]) for g in TOP5] + [(-1, "기타")]
def sgroup(s): return np.where(np.isin(s, TOP5), s, -1)

def size_of(I):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(I[:, :12], axis=1)
Q5 = np.nanquantile(size_of(I1["P36"]), [0.2, 0.4, 0.6, 0.8])
def qgroup(I):
    m = size_of(I)
    return np.where(np.isnan(m), -1, np.searchsorted(Q5, m, side="right"))
QL = [(i, f"규모 {i + 1}분위" + (" (하위)" if i == 0 else " (상위)" if i == 4 else "")) for i in range(5)]

def share_rows(st, groups, labels, name):
    s = st[:, lo:30]
    out = []
    for g, lab in labels:
        m = groups == g
        if m.sum() == 0:
            continue
        tot = s[m].size
        out.append({"표본": name, "집단": lab, "기업 수": cl.sup(m.sum()), "판정 가능 %": round(100 * (s[m] == 0).sum() / tot, 1),
                    "(a) %": round(100 * (s[m] == 1).sum() / tot, 1), "(b) %": round(100 * (s[m] == 2).sum() / tot, 1), "(c) %": round(100 * (s[m] == 3).sum() / tot, 1)})
    return out

rows = []
for name in ("P36", "P35+"):
    st = APPLY[name]["status"]
    for e in range(lo, 30):
        col = st[:, e]
        rows.append({"표본": name, "기준월": int(panel.months[e]), "판정 가능 %": round(100 * (col == 0).mean(), 1), "(a) %": round(100 * (col == 1).mean(), 1),
                     "(b) %": round(100 * (col == 2).mean(), 1), "(c) %": round(100 * (col == 3).mean(), 1)})
cl.show_table(pd.DataFrame(rows), "판정 상태: 월별", OUT / "nb04", "status_by_month")

rows = []
for name in ("P36", "P35+"):
    rows += share_rows(APPLY[name]["status"], sgroup(sec[name]), SG, name)
cl.show_table(pd.DataFrame(rows), "판정 상태: 업종군별 (P36 기업 수 상위 5개 + 기타)", OUT / "nb04", "status_by_sector")

rows = []
for name in ("P36", "P35+"):
    rows += share_rows(APPLY[name]["status"], qgroup(I1[name]), QL, name)
cl.show_table(pd.DataFrame(rows), "판정 상태: 규모(2023년 평균 I1 5분위, 경계는 P36)별", OUT / "nb04", "status_by_size")

### 적용 범위: 활동량 하한으로 제외된 기업은 어떤 기업인가 (P36)
하한 때문에 한 번도 판정되지 못한 기업과 판정된 기업을 **탐지 이전의 특성**으로 비교합니다. 이 결과의 적용 대상은 '하한 이상 기업'으로 한정됩니다.

In [ ]:
a36 = APPLY["P36"]
ever_def = (~np.isnan(a36["change"].r[:, lo:30])).any(axis=1)
ever_ok = (a36["status"][:, lo:30] == 0).any(axis=1)
grp = np.where(ever_ok, "판정 가능 (하한 이상)", np.where(ever_def, "하한 때문에 제외", "변화율 정의 불가"))
codes5 = np.stack([panel.cnt[c][POP["P36"]] for c in cl.CH5])
ceil23 = ((codes5 == 9) & panel.observed[POP["P36"]][None])[:, :, :12].any(axis=(0, 2))
sz = size_of(I1["P36"])
rows = []
for g in ["판정 가능 (하한 이상)", "하한 때문에 제외", "변화율 정의 불가"]:
    m = grp == g
    if m.sum() == 0:
        continue
    rows.append({"집단": g, "기업 수": cl.sup(m.sum()), "기업 비중 %": round(100 * m.mean(), 1), "2023년 평균 I1 중앙값": round(float(np.nanmedian(sz[m])), 1),
                 "2023년 중 50건초과 채널 보유 %": round(100 * ceil23[m].mean(), 1)})
cl.show_table(pd.DataFrame(rows), "하한 적용 범위 (P36)", OUT / "nb04", "floor_scope")
sg36 = sgroup(sec["P36"])
rows = []
for g, lab in SG:
    row = {"업종군": lab}
    for gg in ["판정 가능 (하한 이상)", "하한 때문에 제외"]:
        m = grp == gg
        row[f"{gg} (%)"] = round(100 * (sg36[m] == g).sum() / max(m.sum(), 1), 1) if m.sum() >= 5 else "-"
    rows.append(row)
cl.show_table(pd.DataFrame(rows), "업종군 구성: 판정 가능 기업 vs 하한 제외 기업", OUT / "nb04", "floor_scope_sector")

## 7. 인터넷 천장 보유 여부별 탐지 (P2 결정)
**첫 판정 가능월의 비교 창**에 50건초과(코드 9) 채널이 하나라도 있었던 기업을 '천장 보유'로 봅니다 (탐지 결과와 무관하게 정의). 천장 보유 기업은 I1이 cap 가정과 천장 지속에 좌우됩니다.

In [ ]:
rows = []
for name in ("P36", "P35+"):
    a = APPLY[name]
    f = dc.ceiling_at_first_judgeable(panel, POP[name], a["change"], PHI)
    jf = (a["status"][:, lo:30] == 0).any(axis=1)
    for val, lab in ((True, "천장 보유 (비교 창에 50건초과 채널)"), (False, "천장 없음")):
        m = jf & (f == val)
        n1 = int((a["det"][m] >= 0).sum())
        rows.append({"표본": name, "집단": lab, "판정 가능 기업": cl.sup(m.sum()), "탐지 기업": cl.sup(n1), "탐지율 %": round(100 * n1 / max(int(m.sum()), 1), 1)})
cl.show_table(pd.DataFrame(rows), "천장 보유 여부별 탐지율", OUT / "nb04", "detect_by_ceiling")

## 8. 검정력과 확증 조건 (계획서 4d)
확증 보고 조건(운영 기준, 통계적 보장 아님): **N1 ≥ 100 이고 MDE ≤ 3.0**. 충족하지 못하면 H-C1은 **탐색적으로만** 보고합니다. 실제 CI 폭은 P5의 군집 부트스트랩에서 확인합니다.

In [ ]:
rows, power = [], {}
for name in POP:
    a = APPLY[name]
    jf = int((a["status"][:, lo:30] == 0).any(axis=1).sum()); n1 = int((a["det"] >= 0).sum()); n0 = max(jf - n1, 0)
    m_ = dc.mde(n1, n0)
    ok = (n1 >= 100) and (m_ <= 3.0)
    power[name] = {"N1": n1, "N0": n0, "mde": None if m_ == float("inf") else float(m_), "confirmatory": bool(ok)}
    row = {"표본": name, "N1": cl.sup(n1), "N0": cl.sup(n0), "MDE": "검출 불가" if m_ == float("inf") else round(m_, 2),
           "확증 조건": "충족" if ok else "미충족 → 탐색적 보고"}
    for rr in (1.5, 2, 3):
        if n1 >= 1 and n0 >= 1:
            lo_, hi_, ev = dc.expected_ci(n1, n0, rr)
            row[f"RR={rr}: 기대 약화 사건 수"] = round(ev, 1); row[f"RR={rr}: 기대 95% CI"] = f"{lo_:.2f}~{hi_:.2f}"
    rows.append(row)
cl.show_table(pd.DataFrame(rows), "검정력 근사 (대조군 약화율 5%, 설계 효과 1.5 가정)", OUT / "nb04", "power")

## 9. 규칙·결과 저장 (행 단위 결과는 공유·커밋 금지)

In [ ]:
def _f(x):
    return None if x is None or (isinstance(x, float) and not np.isfinite(x)) else float(x)

spec = {
    "rule": {"비교 기준": MODE, "k": int(K), "L": int(L), "phi": _f(PHI), "n_pct": _f(N), "n0_pct": _f(RULE["n_info"]["n0"]), "q05": _f(RULE["n_info"]["q05"]),
             "exact_share": _f(RULE["n_info"]["exact_share"]), "seasonal": bool(RULE["sea"]["flag"]), "k_why": RULE["k_why"],
             "candidate_months": [int(panel.months[e]) for e in dc.e_range(K)]},
    "decisions": {name: {"mode": d["mode"], "k": int(d["k"]), "phi": _f(d["phi"]), "n_pct": _f(d["n"]), "seasonal": bool(d["sea"]["flag"])} for name, d in DEC.items()},
    "power": power, "synthetic": bool(panel.synthetic),
}
cl.save_json(WORK / "interim" / "detect_spec.json", spec)

# 행 단위 결과(P5 이후 단계가 읽음): P35+ 기준 배열. M1 은 P35만 부분만 채움(P36 은 M0 와 같다).
p35p_idx = np.flatnonzero(POP["P35+"])
pos35 = np.flatnonzero(POP["P35만"][POP["P35+"]])
det_m1_p35p = APPLY["P35+"]["det"].copy()
det_m1_p35p[pos35] = det1
np.savez_compressed(WORK / "interim" / "detect_result.npz", idx_P35p=p35p_idx, det_P35p=APPLY["P35+"]["det"], det_P35p_m1=det_m1_p35p,
                    status_P35p=APPLY["P35+"]["status"], rule=np.array(json.dumps(spec["rule"], ensure_ascii=False)))
print(json.dumps(spec["rule"], ensure_ascii=False, indent=2))
print(json.dumps(power, ensure_ascii=False, indent=2))

## 체크포인트 (Claude에게 알려줄 것)
- [ ] **첫 표(4d)**: P36의 선택된 (비교 기준, k) 행의 판정 가능 기업-월, (a)·(b)·(c) 비율, N1, N0, MDE. 다른 k·방식과 크게 다른 점
- [ ] **① 계절성**(상관 평균·최대/최소 평균·있음/없음), **② 비교 기준**(월별 탐지율 최대/최소, 선택된 방식)
- [ ] **③ 창 길이 k**: 일시적 하락 비율(k별), 선택된 k와 이유, 20% 경계 근처인지
- [ ] **④ Φ**와 하한 제외 비율, Φ의 재표집 분포(중앙값·5~95분위)
- [ ] **⑥ n₀·n**, r이 정확히 −n%인 비율, n±5%p 에서의 탐지 기업 수
- [ ] 세 표본(P36 · P35만 · P35+)의 **결정이 같은가** (비교 기준·k·n)
- [ ] 탐지 기업 수 N1 과 **확증 조건**(N1 ≥ 100 and MDE ≤ 3.0) 충족 여부
- [ ] 판정 불가율: P35만의 (c), 월·업종군·규모별로 눈에 띄는 차이
- [ ] 하한 제외 기업의 특성 (판정 가능 기업과의 차이)
- [ ] 천장 보유 여부별 탐지율
- [ ] M1 vs M0: 최초 탐지월까지 일치 %, 판정 변경률

**다음**: `P4` 판정 기록 → 승인 후 `P5`(입금 검증) 노트북 생성.

## 10. 진단: 결과가 천장(50건초과)에 좌우되는가 (해석용, 사전 규칙이 아님)
위 결과에서 **활동량 하한 Φ가 I1의 cap 값(60) 근처**에서 정해지고 판정 가능 기업 대부분이 천장 보유 기업입니다. 인터넷이 50건초과 구간에 **고정**된 기업은 I1이 cap 값 근처에서 거의 변하지 않아 변화율의 IQR이 매우 작아지고, 이 인공물이 규칙의 기준(상위 분위 IQR)을 끌어내린 것으로 보입니다. 아래는 이를 확인하는 진단입니다. **입금은 쓰지 않습니다.**
- 10-1: 천장 채널이 있는 칸과 없는 칸의 변화율 분포 · 10-2: 탐지가 천장 채널의 하락으로 설명되는 정도 · 10-3: cap(51·60·100)과 하한 규칙(등록 / 천장 칸 제외)에 따른 탐지의 변화

In [ ]:
# 10-1. 천장 채널이 있는 칸 / 없는 칸의 변화율 분포 (P36, 고정 규칙의 k·비교 기준)
import importlib; importlib.reload(dc)   # 모듈이 갱신됐으면 다시 불러온다 (이미 실행한 커널에서도 동작)
lo = K + 11
FLAG = dc.ceiling_window_flag(panel, POP["P36"], K, L)           # (n, 36): 최근·비교 창에 50건초과 채널이 있었는가
ch = APPLY["P36"]["change"]
Cw, rw = ch.C[:, lo:30], ch.r[:, lo:30]
fl = FLAG[:, lo:30]
okp = ~np.isnan(rw) & (Cw > 0)
print(f"변화율이 정의된 기업-월 {int(okp.sum()):,}개 중 천장 채널이 있는 칸 {100 * (okp & fl).sum() / okp.sum():.1f}%")
rows = []
for lab, m in (("천장 채널 없음", okp & ~fl), ("천장 채널 있음", okp & fl)):
    c_, r_ = Cw[m], rw[m]
    qs = np.quantile(c_, [0, .2, .4, .6, .8, 1.0])
    for j in range(5):
        sel = (c_ >= qs[j]) & ((c_ < qs[j + 1]) if j < 4 else (c_ <= qs[j + 1]))
        rows.append({"집단": lab, "C 5분위": j + 1, "C 범위": f"{qs[j]:.1f} ~ {qs[j + 1]:.1f}", "기업-월": cl.sup(sel.sum()),
                     "변화율 중앙값": round(float(np.median(r_[sel])), 3), "IQR": round(float(np.subtract(*np.percentile(r_[sel], [75, 25]))), 3),
                     "5번째 백분위수": round(float(np.percentile(r_[sel], 5)), 3)})
cl.show_table(pd.DataFrame(rows), "천장 채널 유무별, 비교 평균 C 5분위별 변화율 분포 (P36)", OUT / "nb04", "diag_ceiling_iqr")

In [ ]:
# 10-2. 등록 규칙의 탐지가 '천장 채널의 하락'으로 설명되는 정도
det36 = APPLY["P36"]["det"]
share = dc.decompose_detection(panel, POP["P36"], det36, K, L)
idx = np.flatnonzero(det36 >= 0)
r_at = ch.r[idx, det36[idx]]
cat = np.where(share == -1, "천장 채널 없음 (비교 창에 50건초과 채널이 절반 미만)",
      np.where(np.isnan(share), "해당 없음", np.where(share >= 0.8, "천장 채널의 변화가 하락의 80% 이상",
      np.where(share >= 0.5, "천장 채널의 변화가 하락의 50~80%", "천장 채널의 변화가 하락의 50% 미만"))))
rows = []
for c_ in ["천장 채널 없음 (비교 창에 50건초과 채널이 절반 미만)", "천장 채널의 변화가 하락의 80% 이상", "천장 채널의 변화가 하락의 50~80%", "천장 채널의 변화가 하락의 50% 미만", "해당 없음"]:
    m = cat == c_
    if m.sum() == 0:
        continue
    rows.append({"분류": c_, "탐지 기업": cl.sup(m.sum()), "비중 %": round(100 * m.mean(), 1), "탐지월 변화율 중앙값": round(float(np.median(r_at[m])), 3)})
cl.show_table(pd.DataFrame(rows), f"등록 규칙의 탐지 기업 {len(idx):,}개: 총 활동량 하락의 출처 (P36)", OUT / "nb04", "diag_detect_source")

In [ ]:
# 10-3. cap 과 하한 규칙에 따른 탐지의 변화 (고정한 비교 기준·k 에서 ④⑥ 을 다시 수행)
flag_full = FLAG
rows, dets = [], {}
def run_variant(name, cap, alt_floor):
    I_ = I1["P36"] if cap == cl.DEFAULT_CAP else dc.i1_matrix(panel, POP["P36"], cap)
    d_ = dc.decide_fixed(I_, MODE, K, flag_full if alt_floor else None)
    ch_ = d_["change"]; phi_, n_ = d_["phi"], d_["n"]
    det_ = dc.first_detect(ch_, phi_, n_ / 100.0)
    st_ = dc.status_matrix(ch_, phi_)[:, lo:30]
    jf = (st_ == 0).any(axis=1)
    f_ = dc.ceiling_at_first_judgeable(panel, POP["P36"], ch_, phi_)
    dets[name] = det_ >= 0
    return {"변형": name, "cap": int(cap), "하한 규칙": "천장 칸 제외" if alt_floor else "등록", "Φ": round(phi_, 2), "판정 가능 기업": cl.sup(jf.sum()),
            "그중 천장 보유 %": round(100 * f_[jf].mean(), 1) if jf.any() else None, "n (%)": n_, "탐지 기업 N1": cl.sup((det_ >= 0).sum()),
            "탐지율 %": round(100 * (det_ >= 0).sum() / max(int(jf.sum()), 1), 1)}
for cap in (60, 51, 100):
    for alt in (False, True):
        rows.append(run_variant(f"cap {cap} / " + ("대안 하한" if alt else "등록 하한"), float(cap), alt))
base = dets["cap 60 / 등록 하한"]
for r_ in rows:
    o = dets[r_["변형"]]
    r_["등록 규칙 탐지와 겹침 (Jaccard)"] = round(float((o & base).sum() / max((o | base).sum(), 1)), 2)
cl.show_table(pd.DataFrame(rows), "cap × 하한 규칙별 결정과 탐지 (P36, 비교 기준·k 고정)", OUT / "nb04", "diag_variants")

### 진단 체크포인트 (Claude에게 알려줄 것)
- [ ] 10-1: 천장 채널이 **있는 칸 비율**, 두 집단의 **IQR·5번째 백분위수**가 C 5분위별로 어떻게 다른가
- [ ] 10-2: 등록 규칙 탐지 기업 중 **천장 채널의 변화로 설명되는 비중**(80% 이상 / 50~80% / 50% 미만 / 천장 채널 없음)과 각 분류의 탐지월 변화율 중앙값
- [ ] 10-3: 변형별 **Φ · 판정 가능 기업 · 천장 보유 비중 · n · N1 · 겹침(Jaccard)** — 특히 cap 에 따라 탐지가 얼마나 바뀌는지, 대안 하한이 천장 보유 비중을 얼마나 낮추는지

## 11. 수정 규칙 v2: 활동량 하한 산출에서 천장 칸을 제외 (입금 열람 전에 I1 분포만으로 정한 수정)
**진단(§10)에서 확인된 것**: 천장(50건초과) 채널이 있는 칸은 I1이 cap 값으로 기계적으로 고정되어 변화율 IQR이 인공적으로 작아집니다. 이 칸의 IQR이 하한 기준(상위 분위 IQR)을 끌어내려 Φ가 cap 값 근처에서 정해졌고, 판정 가능 기업이 천장 보유 기업으로 쏠렸으며, n(15%)이 '50건 선 아래로 내려오는 정도'의 변화 크기가 되었습니다.

**v2 = 등록 규칙(v1)과 같되, 하한 Φ를 정할 때 분위별 IQR을 '최근 창과 비교 창에 50건초과 채널이 없는 칸'에서만 구합니다.** 정해진 Φ는 모든 칸(천장 칸 포함)에 적용하고, n은 판정 가능 칸 전체의 5번째 백분위수 그대로입니다. 비교 기준(직전 기간)과 k(6)는 ①~③에서 이미 정해졌고 Φ와 무관하므로 그대로입니다.
- v1은 **민감도**로 남깁니다 (P7). P5(입금 검증)의 주 결과는 v2를 씁니다. 이 수정은 입금 결과를 보기 전에, I1 분포만 보고 한 것입니다.

In [ ]:
# 11-1. v2 결정과 적용 (P36 에서 정하고 P35만·P35+ 에는 같은 Φ, n 을 적용)
import importlib; importlib.reload(dc)
lo = K + 11
FL = {name: dc.ceiling_window_flag(panel, m, K, L) for name, m in POP.items()}
D2 = {name: dc.decide_fixed(I1[name], MODE, K, FL[name]) for name in POP}      # 표본 자체의 결정 (비교용)
R2 = D2["P36"]
PHI2, N2 = R2["phi"], R2["n"]
APPLY2 = {name: dc.apply_rule(I1[name], MODE, K, PHI2, N2) for name in POP}
print(f"v2 고정 규칙 (P36): 비교 기준 = {MODE}, k = {K}, Φ = {PHI2:.2f}, n = {N2:.0f}%   (v1: Φ = {PHI:.2f}, n = {N:.0f}%)")
rows, power2 = [], {}
for name in POP:
    a = APPLY2[name]
    st_ = a["status"][:, lo:30]; jf = (st_ == 0).any(axis=1); n1 = int((a["det"] >= 0).sum()); n0 = max(int(jf.sum()) - n1, 0)
    f_ = dc.ceiling_at_first_judgeable(panel, POP[name], a["change"], PHI2)
    m_ = dc.mde(n1, n0); ok = (n1 >= 100) and (m_ <= 3.0)
    power2[name] = {"N1": n1, "N0": n0, "mde": None if m_ == float("inf") else float(m_), "confirmatory": bool(ok)}
    rows.append({"표본": name, "표본 자체의 Φ": round(D2[name]["phi"], 2), "표본 자체의 n (%)": D2[name]["n"], "판정 가능 기업": cl.sup(jf.sum()),
                 "판정 가능 %": round(100 * jf.mean(), 1), "그중 천장 보유 %": round(100 * f_[jf].mean(), 1) if jf.any() else None,
                 "탐지 기업 N1": cl.sup(n1), "탐지율 %": round(100 * n1 / max(int(jf.sum()), 1), 1),
                 "MDE": "검출 불가" if m_ == float("inf") else round(m_, 2), "확증 조건": "충족" if ok else "미충족 → 탐색적"})
cl.show_table(pd.DataFrame(rows), "v2: 고정 규칙(P36의 Φ·n)을 세 표본에 적용", OUT / "nb04", "v2_counts")
print("P35만·P35+ 가 자기 표본에서 정한 Φ·n 도 위 표에 함께 있습니다 (결정 일치 확인용).")

In [ ]:
# 11-2. v2 탐지의 출처, 규모 5분위·천장 유무별 탐지율 (v1 과 비교)
det2 = APPLY2["P36"]["det"]; ch2 = APPLY2["P36"]["change"]
share2 = dc.decompose_detection(panel, POP["P36"], det2, K, L)
idx2 = np.flatnonzero(det2 >= 0); r_at2 = ch2.r[idx2, det2[idx2]]
cat2 = np.where(share2 == -1, "천장 채널 없음", np.where(np.isnan(share2), "해당 없음", np.where(share2 >= 0.8, "천장 채널의 변화가 하락의 80% 이상",
       np.where(share2 >= 0.5, "천장 채널의 변화가 하락의 50~80%", "천장 채널의 변화가 하락의 50% 미만"))))
rows = []
for c_ in ["천장 채널 없음", "천장 채널의 변화가 하락의 80% 이상", "천장 채널의 변화가 하락의 50~80%", "천장 채널의 변화가 하락의 50% 미만", "해당 없음"]:
    m = cat2 == c_
    if m.sum():
        rows.append({"분류": c_, "탐지 기업": cl.sup(m.sum()), "비중 %": round(100 * m.mean(), 1), "탐지월 변화율 중앙값": round(float(np.median(r_at2[m])), 3)})
cl.show_table(pd.DataFrame(rows), f"v2 탐지 기업 {len(idx2):,}개: 총 활동량 하락의 출처 (P36)", OUT / "nb04", "v2_detect_source")

q36 = qgroup(I1["P36"])
rows = []
for ver, a in (("v1(등록)", APPLY["P36"]), ("v2(수정)", APPLY2["P36"])):
    jf = (a["status"][:, lo:30] == 0).any(axis=1); d_ = a["det"] >= 0
    for g, lab in QL:
        m = jf & (q36 == g)
        rows.append({"버전": ver, "집단": lab, "판정 가능 기업": cl.sup(m.sum()), "탐지 기업": cl.sup((d_ & m).sum()), "탐지율 %": round(100 * (d_ & m).sum() / max(int(m.sum()), 1), 1)})
    f_ = dc.ceiling_at_first_judgeable(panel, POP["P36"], a["change"], PHI if ver.startswith("v1") else PHI2)
    for val, lab in ((True, "천장 보유"), (False, "천장 없음")):
        m = jf & (f_ == val)
        rows.append({"버전": ver, "집단": lab, "판정 가능 기업": cl.sup(m.sum()), "탐지 기업": cl.sup((d_ & m).sum()), "탐지율 %": round(100 * (d_ & m).sum() / max(int(m.sum()), 1), 1)})
cl.show_table(pd.DataFrame(rows), "규모 5분위(2023년 평균 I1)·천장 유무별 탐지율: v1 vs v2 (P36)", OUT / "nb04", "v2_detect_by_group")

In [ ]:
# 11-3. 탐지 집합 간 겹침: 등록(v1)/수정(v2) 하한 × cap 51·60·100  (cap 가정에 얼마나 의존하는가)
names = list(dets)
mat = pd.DataFrame([[round(float((dets[a_] & dets[b_]).sum() / max((dets[a_] | dets[b_]).sum(), 1)), 2) for b_ in names] for a_ in names], index=names, columns=names)
cl.show_table(mat.reset_index().rename(columns={"index": "변형"}), "탐지 집합 간 겹침 (Jaccard, P36)", OUT / "nb04", "v2_overlap_matrix")
rows = [{"변형": n_, "탐지 기업": cl.sup(dets[n_].sum())} for n_ in names]
print("대안 하한(v2)의 cap 간 겹침이 등록 하한(v1)의 cap 간 겹침(0.5 안팎)보다 높으면 cap 의존이 줄어든 것입니다.")

In [ ]:
# 11-4. v2 에서의 M1(인과적 채움) 민감도 — P35만, 그리고 첫 표(4d)를 v2 하한으로 다시 낸다
t0 = time.time()
det0 = APPLY2["P35만"]["det"]
det1b = dc.first_detect_m1(panel, POP["P35만"], K, L, PHI2, N2 / 100.0)
d0, d1 = det0 >= 0, det1b >= 0
either = d0 | d1
same_month = (det0 == det1b) & d0 & d1
changed = (d0 != d1) | (d0 & d1 & (det0 != det1b))
tab = pd.DataFrame([{"표본": "P35만 (v2)", "기업 수": cl.sup(len(det0)), "M0 탐지 기업": cl.sup(d0.sum()), "M1 탐지 기업": cl.sup(d1.sum()),
                     "둘 중 하나라도 탐지": cl.sup(either.sum()), "최초 탐지월까지 일치 %": round(100 * same_month.sum() / max(either.sum(), 1), 1),
                     "판정 변경률 % (전체 기업 중)": round(100 * changed.mean(), 1)}])
cl.show_table(tab, "M0 vs M1 (v2, P35만)", OUT / "nb04", "v2_m1_vs_m0")
t4d2 = pd.concat([dc.first_table(I1["P36"], "P36 (v2)", ceil_flag_fn=lambda k_, L_: dc.ceiling_window_flag(panel, POP["P36"], k_, L_)),
                  dc.first_table(I1["P35만"], "P35만 (v2)", ceil_flag_fn=lambda k_, L_: dc.ceiling_window_flag(panel, POP["P35만"], k_, L_))], ignore_index=True)
cl.show_table(_fmt(t4d2), "v2: k별·비교 방식별 판정 가능 월·사건 수·검정력 (4d)", OUT / "nb04", "v2_first_table_4d")
print(f"({time.time() - t0:.0f}초)")

In [ ]:
# 11-5. v2 규칙·결과 저장 (P5 이후 단계는 v2 를 주 결과로 읽는다; 행 단위 결과는 공유·커밋 금지)
def _f2(x):
    return None if x is None or (isinstance(x, float) and not np.isfinite(x)) else float(x)
spec2 = {"version": "v2", "rule": {"비교 기준": MODE, "k": int(K), "L": int(L), "phi": _f2(PHI2), "n_pct": _f2(N2), "n0_pct": _f2(R2["n_info"]["n0"]),
                                  "q05": _f2(R2["n_info"]["q05"]), "exact_share": _f2(R2["n_info"]["exact_share"]),
                                  "floor_excludes_ceiling_cells": True, "candidate_months": [int(panel.months[e]) for e in dc.e_range(K)]},
         "v1_rule": {"phi": _f2(PHI), "n_pct": _f2(N)}, "power": power2, "synthetic": bool(panel.synthetic)}
cl.save_json(WORK / "interim" / "detect_spec_v2.json", spec2)
det_m1_v2 = APPLY2["P35+"]["det"].copy()
det_m1_v2[np.flatnonzero(POP["P35만"][POP["P35+"]])] = det1b
np.savez_compressed(WORK / "interim" / "detect_result_v2.npz", idx_P35p=np.flatnonzero(POP["P35+"]), det_P35p=APPLY2["P35+"]["det"], det_P35p_m1=det_m1_v2,
                    status_P35p=APPLY2["P35+"]["status"], rule=np.array(json.dumps(spec2["rule"], ensure_ascii=False)))
print(json.dumps(spec2["rule"], ensure_ascii=False, indent=2)); print(json.dumps(power2, ensure_ascii=False, indent=2))

### v2 체크포인트 (Claude에게 알려줄 것)
- [ ] 11-1: v2의 **Φ · n · 판정 가능 기업(%) · 그중 천장 보유 % · N1 · MDE · 확증 조건**, 그리고 P35만·P35+가 자기 표본에서 정한 Φ·n 과의 일치
- [ ] 11-2: v2 탐지의 출처 분류, 규모 5분위·천장 유무별 탐지율(v1 vs v2)
- [ ] 11-3: 겹침 행렬 — 특히 **v2의 cap 간 겹침**(cap 60 vs 51, cap 60 vs 100)이 v1(0.48~0.53)보다 높은가
- [ ] 11-4: M1 vs M0(v2), v2 첫 표(4d)에서 선택된 (직전, k=6) 행의 판정 가능 기업-월과 MDE